In [2]:
import pandas as pd
DDI = pd.read_csv("DDI_sea.csv")
ea = pd.read_csv("efectos_adversos.csv")
DDI.head()

,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion,Prioridad
0,DB00001,B01AE02,Lepirudin,target,F2,inhibitor,1.0
1,DB00002,L01FE01,Cetuximab,target,EGFR,binder,1.0
2,DB00002,L01FE01,Cetuximab,target,FCGR3B,binder,1.0
3,DB00002,L01FE01,Cetuximab,target,C1QA,binder,1.0
4,DB00002,L01FE01,Cetuximab,target,C1QB,binder,1.0


In [ ]:
ppio1 = "clopidogrel"
ppio2 = "omeprazole"

In [ ]:
# Pasos previos ¿o lo hago con lower?
DDI["ppio_normalizado"] = DDI["Drug_name"].str.strip().str.casefold()
ppio1 = ppio1.strip().casefold()
ppio2 = ppio2.strip().casefold()

In [ ]:
#Cambio en interaccion: Este sera el principal que saque el diccionario adecuado
def interaccion (ppio_1, ppio_2, DDI, texto=False):
    '''
    Funcion que determina si existe interacción entre los dos principios activos o no.
    Ademas si texto, describe: cada principio y sus enzimas, 
    
    Parámetros
    -----------------
        ppio_1 - El primer principio activo que se quiere comparar con el segundo
        ppio_2 - El segundo principio activo que se quiere comparar con el primero
        DDI - Dataframe que contiene los nombres, ATC, enzimas, target, acciones y prioridad
        texto - Si es necesaria la impresión de texto o no (para comparar el ATC no lo es), por defecto a False
    Devolucion
    ------------------
        String con el nivel de riesgo Leve, Medio y Alto
    '''
    
    #Comprobamos primero que esté en nuestra BBDD
    if (ppio_1 in DDI["ppio_normalizado"].values) and (ppio_2 in DDI["ppio_normalizado"].values):
        df_1 = DDI[DDI["ppio_normalizado"] == ppio_1]
        enz_1, ppal_1 = principales(df_1)
        
        df_2 = DDI[DDI["ppio_normalizado"] == ppio_2]
        enz_2, ppal_2 = principales(df_2)
    else: 
        if (ppio_1 in DDI["Drug_name"].values):
            print(f"{ppio_1} no se ha encontrado en la base de datos, porfavor introduce otro")
            #Dar posibles opciones??
        elif (ppio_2 in DDI["Drug_name"].values):
            print(f"{ppio_2} no se ha encontrado en la base de datos, porfavor introduce otro")
            #Dar posibles opciones?
            
    #Sabiendo que está, no queremos texto
    intro_1 = texto_intro(ppio_1, enz_1, ppal_1) 
    intro_2 = texto_intro(ppio_2, enz_2, ppal_2)



    #Comparamos con las ppales o con la lista de enzimas
    if intro_1:
        comparo1 = ppal_1
    else:
        comparo1 = enz_1
    if intro_2:
        comparo2 = ppal_2
    else:
        comparo2 = enz_2
    #Aqui compruebo si hay interaccion o no
    coincidentes = list(set(comparo1) & set(comparo2))
    #Calculamos el riesgo y si se desea texto imprime el nivel de riesgo que tendrán
    riesgo = calcular_riesgo(ppio_1, ppio_2,coincidentes, intro_1, intro_2)

    return riesgo

In [ ]:
#Esto en el aapartado de acciones
#Si hay interaccion entre ellas se mostrará en coincidentes, sino, la lista será vacia
    if coincidentes:
        #Sacar las acciones y el texto sera para cada enzima por separado
        if texto:
            for e in coincidentes:
                #La fila q contiene la enzima que se quiere ver
                fila_1 = df_1[df_1["Gene_name"]==e]
                #Separamos por si tiene | (no da error si no lo tiene)
                separado_1 = fila_1["Accion"].str.split(r"\|")
                #Nos quedamos con los distintos
                acciones_1 = set(separado_1.explode().tolist())
    
                #Lo mismo pero con el otro principio consultado
                fila_2 = df_2[df_2["Gene_name"]==e]
                separado_2 = fila_2["Accion"].str.split(r"\|")
                acciones_2 = set(separado_2.explode().tolist())

                #Funcion que contiene el texto
                texto_acciones(ppio_1,acciones_1,ppio_2,acciones_2,e) #PARA CADA ENZIMA